# 07 · Dropout

**Belgedeki bölüm:** Dropout

- Eğitim sırasında attention ağırlıklarının bir kısmı rastgele sıfırlanır; model tek bir kelimeye aşırı güvenmez.
- İki yerde uygulanır: softmax'tan sonra **attention ağırlıklarında** ve **context vector çıkışında**.
- Faydaları: ezberlemeyi önler, daha dayanıklı temsiller, kapasiteyi kullanır.
- Sadece eğitimde açıktır; `eval()` modunda kapanır. PyTorch kalan değerleri `1/(1-p)` ile ölçekler.

In [ ]:
import torch

torch.manual_seed(1)
attention_scores = torch.randn(7, 7)

mask = torch.tril(torch.ones(7, 7))
masked_attention_scores = attention_scores.masked_fill(mask == 0, -torch.inf)

softmaxed_attention_weights = torch.softmax(masked_attention_scores, dim=1)
softmaxed_attention_weights

### dropout_rate = 0 → hiçbir şey değişmez

In [ ]:
dropout_rate = 0
torch.manual_seed(1)
dropout = torch.nn.Dropout(dropout_rate)
dropout(softmaxed_attention_weights)

### dropout_rate = 0.5 → değerlerin yaklaşık yarısı 0 olur, kalanlar 1 / (1 − 0.5) = 2 ile çarpılır

In [ ]:
dropout = torch.nn.Dropout(0.5)

torch.manual_seed(1)
print(dropout(softmaxed_attention_weights))

# Her çağrıda farklı bağlantılar kapanır
print(dropout(softmaxed_attention_weights))

### Sadece eğitimde: `eval()` modunda dropout kapalıdır

In [ ]:
dropout.eval()
dropout(softmaxed_attention_weights)  # değerler aynen geçer

### Modelde: UstaCausalAttention(dropout_rate=...)
Dropout, softmax'tan sonra attention ağırlıklarına uygulanır (`usta_causal_attention.py`).

In [ ]:
from usta_causal_attention import UstaCausalAttention

torch.manual_seed(1)
x = torch.randn(5, 4)  # 5 token, her biri 4 boyutlu

causal_attention = UstaCausalAttention(embedding_dim=4, output_dim=4, context_length=32, dropout_rate=0.5)

causal_attention.train()
first, second = causal_attention(x), causal_attention(x)
print("eğitim modu  → iki çağrı aynı mı?", torch.equal(first, second))  # False: her seferinde farklı maske

causal_attention.eval()
first, second = causal_attention(x), causal_attention(x)
print("çıkarım modu → iki çağrı aynı mı?", torch.equal(first, second))  # True: dropout kapalı

### Context vector çıkışında dropout
Belgedeki ikinci nokta: ağırlıklı toplamdan sonra da dropout uygulanabilir (nöronlar birbirine aşırı bağımlı olmasın).

In [ ]:
context_vector = torch.randn(5, 4)

output_dropout = torch.nn.Dropout(0.2)
output_dropout.train()
output_dropout(context_vector)